In [7]:
import torch
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
import os

# --- CONFIGURATION ---
# These parameters MUST match the ones used in the training/generation scripts.
SEQ_LEN = 12      # Input history length
PRED_LEN = 3     # Output forecast horizon
DATA_PATH = '../../../ICL4DT/data/time_series_datasets/ETTm2.csv'
HTI_DATA_DIR = 'hti_data'

# The quantiles of the expert models you want to visualize
QUANTILES_TO_LOAD = [0.01, 0.1, 0.25, 0.5, 0.75,0.9, 0.99]

print("Loading original dataset...")
df = pd.read_csv(DATA_PATH)
data = df['OT'].values.astype(float)

print(f"Full dataset shape: {data.shape}")

# Recreate the exact train/val/test split to fit the scaler correctly
train_split_idx = int(len(data) * 0.7)
val_split_idx = int(len(data) * 0.98)

# Isolate the original, unscaled test data for ground truth comparison
original_test_data = data[val_split_idx:]

# Fit the scaler ONLY on the training data to prevent data leakage
print("Fitting MinMaxScaler on the training data portion...")
scaler = MinMaxScaler(feature_range=(-1, 1))
scaler.fit(data[:train_split_idx].reshape(-1, 1))

hti_datasets = {}
all_forecasts_unscaled = {}

print("Loading HTI datasets and unscaling forecasts...")

for q in QUANTILES_TO_LOAD:
    # Construct filename (e.g., hti_data_q05.pt)
    filename = f"hti_data_q{str(q).replace('.', '')}.pt"
    
    try:
        # Load the entire [history, forecast] tensor
        hti_datasets[q] = torch.load(filename)
        data = hti_datasets[q]
        print(f" -> Loaded '{filename}' with shape: {hti_datasets[q].shape}")
        data_unscaled = scaler.inverse_transform(data)
        data_unscaled = torch.tensor(data_unscaled, dtype=torch.float32)
        all_forecasts_unscaled[q] = data_unscaled
        
    except FileNotFoundError:
        print(f" -> WARNING: Could not find file {filename}. Skipping.")

print("\nForecasts are now unscaled and ready for plotting.")


Loading original dataset...
Full dataset shape: (69680,)
Fitting MinMaxScaler on the training data portion...
Loading HTI datasets and unscaling forecasts...
 -> Loaded 'hti_data_q001.pt' with shape: torch.Size([1380, 15])
 -> Loaded 'hti_data_q01.pt' with shape: torch.Size([1380, 15])
 -> Loaded 'hti_data_q025.pt' with shape: torch.Size([1380, 15])
 -> Loaded 'hti_data_q05.pt' with shape: torch.Size([1380, 15])
 -> Loaded 'hti_data_q075.pt' with shape: torch.Size([1380, 15])
 -> Loaded 'hti_data_q09.pt' with shape: torch.Size([1380, 15])
 -> Loaded 'hti_data_q099.pt' with shape: torch.Size([1380, 15])

Forecasts are now unscaled and ready for plotting.


In [8]:
all_forecasts_unscaled

{0.01: tensor([[34.1700, 34.3895, 35.0485,  ..., 40.7864, 40.9641, 40.8561],
         [34.3895, 35.0485, 35.4885,  ..., 41.9019, 42.1834, 42.1430],
         [35.0485, 35.4885, 36.1475,  ..., 42.7777, 43.1059, 43.0847],
         ...,
         [48.1835, 48.1835, 48.1835,  ..., 45.1789, 44.9654, 44.5007],
         [48.1835, 48.1835, 48.1835,  ..., 44.7118, 44.4696, 43.9851],
         [48.1835, 48.1835, 48.1835,  ..., 44.8542, 44.6519, 44.2335]]),
 0.1: tensor([[34.1700, 34.3895, 35.0485,  ..., 41.0872, 41.1586, 41.4614],
         [34.3895, 35.0485, 35.4885,  ..., 42.3159, 42.4360, 42.8184],
         [35.0485, 35.4885, 36.1475,  ..., 43.1745, 43.2841, 43.6663],
         ...,
         [48.1835, 48.1835, 48.1835,  ..., 45.7739, 45.3420, 45.0736],
         [48.1835, 48.1835, 48.1835,  ..., 45.2637, 44.8167, 44.5314],
         [48.1835, 48.1835, 48.1835,  ..., 45.2852, 44.8791, 44.6300]]),
 0.25: tensor([[34.1700, 34.3895, 35.0485,  ..., 41.4244, 41.6937, 41.9174],
         [34.3895, 35.0485, 

In [9]:
combined = torch.stack([all_forecasts_unscaled[q] for q in QUANTILES_TO_LOAD], dim=0)

torch.save(combined, 'hti_data_combined.pt')

In [10]:
combined.shape

torch.Size([7, 1380, 15])

In [11]:
combined

tensor([[[34.1700, 34.3895, 35.0485,  ..., 40.7864, 40.9641, 40.8561],
         [34.3895, 35.0485, 35.4885,  ..., 41.9019, 42.1834, 42.1430],
         [35.0485, 35.4885, 36.1475,  ..., 42.7777, 43.1059, 43.0847],
         ...,
         [48.1835, 48.1835, 48.1835,  ..., 45.1789, 44.9654, 44.5007],
         [48.1835, 48.1835, 48.1835,  ..., 44.7118, 44.4696, 43.9851],
         [48.1835, 48.1835, 48.1835,  ..., 44.8542, 44.6519, 44.2335]],

        [[34.1700, 34.3895, 35.0485,  ..., 41.0872, 41.1586, 41.4614],
         [34.3895, 35.0485, 35.4885,  ..., 42.3159, 42.4360, 42.8184],
         [35.0485, 35.4885, 36.1475,  ..., 43.1745, 43.2841, 43.6663],
         ...,
         [48.1835, 48.1835, 48.1835,  ..., 45.7739, 45.3420, 45.0736],
         [48.1835, 48.1835, 48.1835,  ..., 45.2637, 44.8167, 44.5314],
         [48.1835, 48.1835, 48.1835,  ..., 45.2852, 44.8791, 44.6300]],

        [[34.1700, 34.3895, 35.0485,  ..., 41.4244, 41.6937, 41.9174],
         [34.3895, 35.0485, 35.4885,  ..., 42

In [12]:
combined_min = combined.min().item()
combined_max = combined.max().item()
print(f"Min value in combined: {combined_min}")
print(f"Max value in combined: {combined_max}")

Min value in combined: 25.62824249267578
Max value in combined: 56.70988464355469
